# MediAI: Advanced Symptom Deductor Training (Cancer & Neurology)

This notebook is optimized for **Google Colab (T4/A100 GPU)** and **Kaggle**. It fine-tunes a medical LLM using specialized oncology and neuroscience datasets.

### Platforms Supported:
- **Google Colab**: 🔑 Add your `HF_TOKEN` to the secrets menu.
- **Kaggle**: 📂 Upload your `kaggle.json` to enable dataset downloads.

In [ ]:
# 1. Install Requirements
!pip install -q -U torch transformers datasets accelerate evaluate peft huggingface_hub kaggle bitsandbytes trl

## 2. Authentication & Environment
We securely load API keys to fetch datasets and models.

In [ ]:
import os
from huggingface_hub import login

try:
    # For Colab users: use userdata secrets
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
except:
    # For local or Kaggle users
    HF_TOKEN = "YOUR_HF_TOKEN_HERE" # Default from backend/.env

if HF_TOKEN:
    login(token=HF_TOKEN)
else:
    print("WARNING: HF_TOKEN not found. You might not be able to download Llama 3.")

## 3. Load Specialized Datasets
We combine general symptom data with specialized cancer (HoC) and brain (BrainGPT) data.

In [ ]:
from datasets import load_dataset, concatenate_datasets

print("Loading specialized medical datasets...")

# 1. Cancer Classification (Hallmarks of Cancer)
hoc_dataset = load_dataset("precision-medicine/hoc", trust_remote_code=True)

# 2. Neuroscience (BrainGPT Training Split)
brain_dataset = load_dataset("BrainGPT/train_valid_split_pmc_neuroscience", split='train')

# 3. General Medical QA (MedQA Reduced for speed)
medqa_dataset = load_dataset("bigbio/med_qa", split='train[:20%]', trust_remote_code=True)

print(f"Loaded {len(hoc_dataset['train'])} cancer records and {len(brain_dataset)} neuroscience records.")

## 4. Model Configuration (PEFT/LoRA)
We use 4-bit quantization to fit the model in free Colab memory.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments
from peft import LoraConfig, get_peft_model
import torch

model_id = "meta-llama/Llama-3-8B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
print("Model ready for 4-bit fine-tuning on specialized medical data.")

## 5. Training Pipeline
We use the `SFTTrainer` for efficient instruction tuning.

In [ ]:
from trl import SFTTrainer

training_args = TrainingArguments(
    output_dir="./mediai-specialist-llama3",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    max_steps=500, # Adjust for full training
    logging_steps=10,
    save_strategy="steps",
    save_steps=100,
    fp16=True,
    report_to="none"
)

trainer = SFTTrainer(
    model=model,
    train_dataset=brain_dataset, # Concatenate datasets for full run
    dataset_text_field="text",
    max_seq_length=512,
    tokenizer=tokenizer,
    args=training_args
)

print("Starting training... Observe the loss curve in the logs below.")
# trainer.train() # Uncomment to run in Colab/Kaggle